In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import rdkit
from scipy import stats

project_root_dir = "/opt/pubchem-cid4-analysis/py/"
if project_root_dir not in sys.path:
    sys.path.append(project_root_dir)
print(sys.path)

from src import log_settings  # noqa: E402
from src.cid4_analysis import (  # noqa: E402
    build_activity_posterior_dataframe,
    build_distance_matrix,
    extract_3d_coordinates,
    get_output_directory,
    load_bioactivity_dataframe,
    load_conformer_compound,
    load_sdf_molecules,
)
from src.constants import ARR_1ST_IDX as IDX1  # noqa: E402
from src.constants import UTF_8  # noqa: E402
from src.utils import env_utils  # noqa: E402

['/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/opt/pubchem-cid4-analysis/py/.venv/lib/python3.12/site-packages', '/opt/pubchem-cid4-analysis/py/']


### Distance matrix

In [ ]:
sdf_filename = "Conformer3D_COMPOUND_CID_4(1).sdf"
json_filename = "Conformer3D_COMPOUND_CID_4(1).json"
molecules = load_sdf_molecules(sdf_filename)
print(f"molecules length={len(molecules)}")
for mol in molecules:
    print(f"mol.GetNumAtoms={mol.GetNumAtoms()}")
    print(f"mol.GetNumBonds={mol.GetNumBonds()}")
    print(f"mol.GetNumConformers={mol.GetNumConformers()}")
    print(f"mol.GetNumHeavyAtoms={mol.GetNumHeavyAtoms()}")

INFO 2026-04-05T11:26:09+0300 env_utils.py get_data_dir 7 root 66906 MainProcess 140704334209856 MainThread Task-3 Data directory: /Users/Artsem_Nikitsenka/projects/pubchem-cid4-analysis/data
molecules length=1
mol.GetNumAtoms=14
mol.GetNumBonds=13
mol.GetNumConformers=1
mol.GetNumHeavyAtoms=5


In [ ]:
coordinates = extract_3d_coordinates(molecules[IDX1])
type(coordinates), coordinates

(numpy.ndarray,
 array([[ 1.5903, -0.8258,  0.0378],
        [-1.9942,  0.1028, -0.1015],
        [ 0.4693, -0.0273, -0.3404],
        [-0.7967, -0.643 ,  0.2606],
        [ 0.7313,  1.3933,  0.1435],
        [ 0.4149, -0.0317, -1.4353],
        [-0.709 , -0.6871,  1.3526],
        [-0.8967, -1.6812, -0.0775],
        [ 1.6841,  1.7603, -0.2541],
        [ 0.8156,  1.4252,  1.2355],
        [-0.0587,  2.0827, -0.1681],
        [-2.8156, -0.377 ,  0.2637],
        [-2.0968,  0.1155, -1.1153],
        [ 1.4396, -1.7233, -0.3047]]))

In [ ]:
distance_matrix = build_distance_matrix(coordinates)
distance_matrix

array([[0.        , 3.70544771, 1.42733265, 2.40433456, 2.38190224,
        2.04503877, 2.65230413, 2.63252203, 2.60421145, 2.66489144,
        3.34977134, 4.43445679, 3.97622833, 0.97238006],
       [3.70544771, 0.        , 2.47847362, 1.45648217, 3.02551905,
        2.75696853, 2.09525365, 2.09469288, 4.03738578, 3.3810186 ,
        2.76958477, 1.0189588 , 1.01905765, 3.89447222],
       [1.42733265, 2.47847362, 0.        , 1.53070033, 1.52345252,
        1.09625943, 2.16563523, 2.16112416, 2.1630304 , 2.17097691,
        2.18187334, 3.35824283, 2.68434928, 1.95427034],
       [2.40433456, 1.45648217, 1.53070033, 0.        , 2.54853175,
        2.17203569, 1.09640326, 1.09643552, 3.49215344, 2.79774776,
        2.8561982 , 2.03635037, 2.03928494, 2.54708655],
       [2.38190224, 3.02551905, 1.52345252, 2.54853175, 0.        ,
        2.15019613, 2.80436286, 3.48593965, 1.09572013, 1.09571351,
        1.0938313 , 3.96596895, 3.34895504, 3.22734685],
       [2.04503877, 2.75696853, 1.0

In [54]:
point: rdkit.Geometry.rdGeometry.Point3D = molecules[IDX1].GetConformer().GetAtomPosition(0)
print(point.x, point.y, point.z)

1.5903 -0.8258 0.0378


In [ ]:
atom_ids = load_conformer_compound(json_filename)["atoms"]["aid"]
atom_ids

INFO 2026-04-05T12:03:03+0300 env_utils.py get_data_dir 7 root 66906 MainProcess 140704334209856 MainThread Task-3 Data directory: /Users/Artsem_Nikitsenka/projects/pubchem-cid4-analysis/data


[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]

# Bioactivity binomial analysis

In [ ]:
DEFAULT_BINOMIAL_TAIL_THRESHOLD = 0.0


def build_assay_activity_binomial_dataframe(
    posterior_df: pd.DataFrame,
    counts: dict[str, int],
) -> tuple[pd.DataFrame, dict[str, int]]:
    assay_df = (
        posterior_df.groupby("BioAssay_AID", dropna=False)
        .agg(
            retained_binary_rows=("Activity", "size"),
            active_rows=("Activity", lambda values: int(values.eq("Active").sum())),
            inactive_rows=("Activity", lambda values: int(values.eq("Inactive").sum())),
            sample_activity_type=("Activity_Type", "first"),
            sample_target_name=("Target_Name", "first"),
            sample_bioassay_name=("BioAssay_Name", "first"),
        )
        .reset_index()
    )
    assay_df["mixed_evidence"] = assay_df["active_rows"].gt(0) & assay_df["inactive_rows"].gt(0)
    assay_df["assay_activity"] = np.where(assay_df["active_rows"].gt(0), "Active", "Inactive")
    assay_df = assay_df.sort_values(by=["assay_activity", "BioAssay_AID"]).reset_index(drop=True)

    assay_counts = {
        **counts,
        "assay_trials": int(len(assay_df)),
        "active_assay_trials": int(assay_df["assay_activity"].eq("Active").sum()),
        "inactive_assay_trials": int(assay_df["assay_activity"].eq("Inactive").sum()),
        "mixed_evidence_assay_trials": int(assay_df["mixed_evidence"].sum()),
        "unanimous_active_assay_trials": int((assay_df["active_rows"] > 0).sum() - assay_df["mixed_evidence"].sum()),
        "unanimous_inactive_assay_trials": int(
            (assay_df["inactive_rows"] > 0).sum() - assay_df["mixed_evidence"].sum()
        ),
    }

    if assay_df.empty:
        raise ValueError("No assay-level Active/Inactive trials were found in the bioactivity dataset")

    return assay_df, assay_counts


def build_bioactivity_binomial_pmf_dataframe(assay_df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, float | int]]:
    assay_trials = int(len(assay_df))
    active_assay_trials = int(assay_df["assay_activity"].eq("Active").sum())
    success_probability = active_assay_trials / assay_trials
    k_values = np.arange(assay_trials + 1, dtype=np.int64)
    pmf_values = stats.binom.pmf(k_values, assay_trials, success_probability)
    cumulative_leq_values = stats.binom.cdf(k_values, assay_trials, success_probability)
    cumulative_geq_values = stats.binom.sf(
        np.maximum(k_values - 1, DEFAULT_BINOMIAL_TAIL_THRESHOLD), assay_trials, success_probability
    )
    pmf_df = pd.DataFrame(
        {
            "k_active": k_values,
            "probability": pmf_values,
            "cumulative_probability_leq_k": cumulative_leq_values,
            "cumulative_probability_geq_k": cumulative_geq_values,
        }
    )

    return pmf_df, {
        "assay_trials": assay_trials,
        "active_assay_trials": active_assay_trials,
        "success_probability_active_assay": float(success_probability),
        "observed_pmf_at_active_assay_count": float(pmf_values[active_assay_trials]),
        "observed_cumulative_probability_leq_active_assay_count": float(cumulative_leq_values[active_assay_trials]),
        "observed_cumulative_probability_geq_active_assay_count": float(cumulative_geq_values[active_assay_trials]),
    }


def summarize_bioactivity_binomial_analysis(
    assay_df: pd.DataFrame,
    pmf_df: pd.DataFrame,
    counts: dict[str, int],
    binomial_metrics: dict[str, float | int],
) -> dict:
    assay_trials = int(binomial_metrics["assay_trials"])
    active_assay_trials = int(binomial_metrics["active_assay_trials"])
    success_probability = float(binomial_metrics["success_probability_active_assay"])
    representative_positions = sorted({0, len(assay_df) // 2, len(assay_df) - 1})
    representative_rows = assay_df.iloc[representative_positions]

    return {
        "row_counts": counts,
        "binomial": {
            "parameters": {
                "n_assays": assay_trials,
                "observed_active_assays": active_assay_trials,
                "success_probability_active_assay": success_probability,
            },
            "summary": {
                "pmf_at_observed_active_assay_count": float(binomial_metrics["observed_pmf_at_active_assay_count"]),
                "cumulative_probability_leq_observed_active_assay_count": float(
                    binomial_metrics["observed_cumulative_probability_leq_active_assay_count"]
                ),
                "cumulative_probability_geq_observed_active_assay_count": float(
                    binomial_metrics["observed_cumulative_probability_geq_active_assay_count"]
                ),
                "binomial_mean_active_assays": float(assay_trials * success_probability),
                "binomial_variance_active_assays": float(
                    assay_trials * success_probability * (1.0 - success_probability)
                ),
                "pmf_probability_sum": float(pmf_df["probability"].sum()),
            },
        },
        "analysis": {
            "representative_assays": [
                {
                    "BioAssay_AID": int(row["BioAssay_AID"]),
                    "BioAssay_Name": str(row["sample_bioassay_name"]),
                }
                for _, row in representative_rows.iterrows()
            ],
        },
    }


def run_bioactivity_binomial_analysis():
    log_settings.configure_logging()

    work_directory = env_utils.get_data_dir()
    out_dir = get_output_directory(work_directory)
    file_name = "pubchem_cid_4_bioactivity.csv"
    bioactivity_df = load_bioactivity_dataframe(file_name)
    print(bioactivity_df)
    posterior_df, posterior_counts = build_activity_posterior_dataframe(bioactivity_df)
    print(posterior_df)
    assay_df, assay_counts = build_assay_activity_binomial_dataframe(posterior_df, posterior_counts)
    pmf_df, binomial_metrics = build_bioactivity_binomial_pmf_dataframe(assay_df)
    print(pmf_df)
    summary = summarize_bioactivity_binomial_analysis(assay_df, pmf_df, assay_counts, binomial_metrics)
    output_stem = Path(file_name).stem
    summary_output_path = Path(out_dir) / f"{output_stem}.activity_binomial.summary.json"

    with summary_output_path.open("w", encoding=UTF_8) as file:
        json.dump(summary, file, indent=2)


run_bioactivity_binomial_analysis()

INFO 2026-05-26T05:39:37+0000 env_utils.py get_data_dir 7 root 82 MainProcess 139857695187712 MainThread Task-2 Data directory: /opt/pubchem-cid4-analysis/py/../data
INFO 2026-05-26T05:39:37+0000 env_utils.py get_data_dir 7 root 82 MainProcess 139857695187712 MainThread Task-2 Data directory: /opt/pubchem-cid4-analysis/py/../data
     Bioactivity_ID      Aid_Type     Activity Protein_Accession  \
0         334754348  Confirmatory  Unspecified          AEP43755   
1         100909607  Confirmatory        Probe               NaN   
2           3211067         Other  Unspecified               NaN   
3           3219009         Other  Unspecified               NaN   
4         383459365  Confirmatory  Unspecified          AAI13547   
..              ...           ...          ...               ...   
401       356503192  Confirmatory  Unspecified               NaN   
402       392415959     Screening  Unspecified               NaN   
403       356544943       Summary  Unspecified          